# Collection Tool-Call Wiring

## Goal

Make collection runs queryable through the shared `llm_client` `tool_calls` backend without waiting for a full fetch-path migration.

This slice is intentionally narrow:
- pass `trace_id` and `task` into `open_web_retrieval` search calls
- emit shared `tool_calls` from the project-local fetch and Jina fallback path
- keep the collection trace queryable under one run-level `trace_id`


## Boundary Decision

Current collection path:
- search: `grounded_research.tools.brave_search` -> `open_web_retrieval`
- fetch/extract: `collect.py` -> local `fetch_page.py` and `jina_reader.py`

So the correct short-term wiring is hybrid:
- shared search observability comes from `open_web_retrieval`
- local fetch observability comes from `collect.py` using the same shared `ToolCallResult` contract

Do not migrate fetch to shared infra in this slice. That is a later architectural change.


## Event Shape

Search rows:
- tool: `open_web_retrieval`
- operation: `search`
- provider: `brave`
- trace_id: root collection trace
- task: `collection.search`

Local fetch rows:
- tool: `grounded_research`
- operation: `fetch_page`
- provider: `local_fetch_page` or `jina_reader`
- trace_id: root collection trace
- task: `collection.fetch`

Fallback visibility rule:
- failed local fetch row for the 403 path
- separate started/succeeded or started/failed row for `jina_reader`


## Acceptance

After one collection run, SQL must be able to show:
- all search calls for the run
- all local fetch attempts for the run
- which URLs fell back to Jina
- duration and failure counts grouped by provider

This slice is complete when a collection run can be debugged from `tool_calls` rows without ad hoc print-only logging.
